# 🪆 Module 2.5: Nested Runs

**Goal**: Organize complex experiments using parent-child run hierarchies.

Nested runs are essential for **hyperparameter sweeps**, **model comparisons**, and **multi-stage pipelines** where you want to group related runs together.

---

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import itertools

# Disable autolog so we can see manual nested logging clearly
mlflow.autolog(disable=True)


# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

#settin the experiment, this should be done before setting the database or tracking URI
mlflow.set_experiment("02_nested_runs")



Tracking URI: sqlite:///c:/Users/sujat/projects/MLFlow_Learn/mlflow.db


2026/06/23 14:27:04 INFO mlflow.tracking.fluent: Experiment with name '02_nested_runs' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/sujat/projects/MLFlow_Learn/Module_02_Tracking/mlruns/6', creation_time=1782239224903, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1782239224903, lifecycle_stage='active', name='02_nested_runs', tags={}, trace_location=None, workspace='default'>

In [2]:
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.2, random_state=42
)
print("✅ Setup complete!")

✅ Setup complete!


## 1. Basic Nested Run Structure

The key is `nested=True` in the child's `start_run()` call.

```
Parent Run: "HPO Sweep"
    ├── Child Run: trial_1
    ├── Child Run: trial_2
    └── Child Run: trial_3
```

In [4]:
# Simple nested run example
with mlflow.start_run(run_name="simple_parent") as parent_run:
    
    # Log parent-level info
    mlflow.set_tag("purpose", "hyperparameter_sweep")
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("num_trials", 3)
    
    # Track best result across children
    best_accuracy = 0
    best_params = {}
    
    # Child runs with different hyperparameters ('-_-')
    configs = [
        {"n_estimators": 50, "max_depth": 3},
        {"n_estimators": 100, "max_depth": 5},
        {"n_estimators": 200, "max_depth": 10},
    ]
    
    for i, params in enumerate(configs):
        # nested=True makes this a child of the current active run
        with mlflow.start_run(run_name=f"trial_{i+1}", nested=True):
            
            mlflow.log_params(params)
            
            model = RandomForestClassifier(**params, random_state=42)
            model.fit(X_train, y_train)
            
            accuracy = accuracy_score(y_test, model.predict(X_test))
            f1 = f1_score(y_test, model.predict(X_test), average='weighted')
            
            mlflow.log_metrics({"accuracy": accuracy, "f1": f1}) #logging
            mlflow.sklearn.log_model(model, artifact_path="model")
            
            print(f"  Trial {i+1}: n_estimators={params['n_estimators']}, "
                  f"max_depth={params['max_depth']} → accuracy={accuracy:.4f}")
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_params = params
    
    # Log best result on the parent
    mlflow.log_metric("best_accuracy", best_accuracy)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})
    
    print(f"\n🏆 Best: accuracy={best_accuracy:.4f} with {best_params}")
    print(f"   Parent run ID: {parent_run.info.run_id}")

2026/06/23 14:31:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Trial 1: n_estimators=50, max_depth=3 → accuracy=1.0000


2026/06/23 14:32:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Trial 2: n_estimators=100, max_depth=5 → accuracy=1.0000


2026/06/23 14:32:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Trial 3: n_estimators=200, max_depth=10 → accuracy=1.0000

🏆 Best: accuracy=1.0000 with {'n_estimators': 50, 'max_depth': 3}
   Parent run ID: d2c7e72428bb458395af6ea077c422cb


## 2. Grid Search with Nested Runs

A more practical example: systematically try all combinations of hyperparameters.

In [5]:
# Define hyperparameter grid
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 10],
    "min_samples_split": [2, 5],
}

# Generate all combinations
keys = param_grid.keys()
combinations = list(itertools.product(*param_grid.values()))

print(f"Grid search: {len(combinations)} combinations")

with mlflow.start_run(run_name="grid_search_sweep") as parent_run:
    
    mlflow.set_tag("purpose", "grid_search")
    mlflow.log_param("total_trials", len(combinations))
    
    results = []
    
    for i, values in enumerate(combinations):
        params = dict(zip(keys, values))
        
        with mlflow.start_run(run_name=f"grid_{i+1:02d}", nested=True):
            mlflow.log_params(params)
            
            model = RandomForestClassifier(**params, random_state=42)
            model.fit(X_train, y_train)
            
            accuracy = accuracy_score(y_test, model.predict(X_test))
            f1 = f1_score(y_test, model.predict(X_test), average='weighted')
            
            mlflow.log_metrics({"accuracy": accuracy, "f1": f1})
            
            results.append({**params, "accuracy": accuracy, "f1": f1})
    
    # Log summary on parent
    results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False)
    best = results_df.iloc[0]
    
    mlflow.log_metric("best_accuracy", best["accuracy"])
    mlflow.log_metric("best_f1", best["f1"])
    
    # Log results table as artifact
    mlflow.log_text(
        results_df.to_string(index=False),
        "grid_search_results.txt"
    )
    
    print(f"\n📊 Grid Search Results (top 5):")
    print(results_df.head().to_string(index=False))
    print(f"\n🏆 Best accuracy: {best['accuracy']:.4f}")

Grid search: 18 combinations

📊 Grid Search Results (top 5):
 n_estimators  max_depth  min_samples_split  accuracy  f1
           50          3                  2       1.0 1.0
           50          5                  2       1.0 1.0
          100         10                  2       1.0 1.0
           50          5                  5       1.0 1.0
           50         10                  2       1.0 1.0

🏆 Best accuracy: 1.0000


## 3. Viewing Nested Runs in the UI

Open the MLFlow UI and look for the parent run. You can:

1. **Expand** the parent run to see all child runs
2. **Compare** child runs side-by-side
3. **Filter** within a parent run's children

The parent run shows the **best results**, while children show **individual trials**.

## 4. Querying Nested Runs Programmatically

In [7]:
# Find all child runs of our grid search
child_runs = mlflow.search_runs(
    experiment_names=["02_nested_runs"],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
    order_by=["metrics.accuracy DESC"]
)

print(f"Found {len(child_runs)} child runs")
print(f"\nTop 5 by accuracy:")

display_cols = [col for col in [
    "tags.mlflow.runName",
    "params.n_estimators",
    "params.max_depth", 
    "params.min_samples_split",
    "metrics.accuracy",
    "metrics.f1"
] if col in child_runs.columns]

child_runs[display_cols].head()

Found 18 child runs

Top 5 by accuracy:


,tags.mlflow.runName,params.n_estimators,params.max_depth,params.min_samples_split,metrics.accuracy,metrics.f1
0,grid_18,200,10,5,1.0,1.0
1,grid_17,200,10,2,1.0,1.0
2,grid_16,200,5,5,1.0,1.0
3,grid_15,200,5,2,1.0,1.0
4,grid_14,200,3,5,1.0,1.0


## 🔑 Key Takeaways

1. **`nested=True`** in `start_run()` creates a child run under the current active parent
2. **Parent runs** should summarize the overall experiment (best metrics, config)
3. **Child runs** contain individual trial details (specific params and metrics)
4. Use nested runs for **HPO sweeps**, **model comparisons**, and **multi-stage experiments**
5. Query children using `tags.mlflow.parentRunId` filter

---

## ➡️ Next: `06_exercises.ipynb` — Practice what you've learned!